# 02 — 手写 StateGraph 理解 ReAct 循环

> **目标**：理解 create_agent() 底层到底做了什么——自己写 ReAct 循环  
> **产出**：手写的 agent → tools → agent 循环，看清每一步消息流转

## 0. 准备

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from src.config.settings import settings
print(f"模型: {settings.MODEL_NAME}")

## 1. 看看 AgentState — Agent 的"共享内存"

`AgentState` 是一个 `TypedDict`，定义了在节点间流转的数据的形状。
这是 LangGraph 和普通函数调用的最大区别：**节点之间不直接传参，而是读写共享状态**。

In [ ]:
from src.agent.state import AgentState

# TypedDict 本身只是"类型标注"，不包含运行逻辑
print(AgentState.__annotations__)
print()
# 关键：messages 字段用了 Annotated[list, add_messages]
# add_messages 是一个 reducer（合并函数），确保消息是"追加"而非"覆盖"
print(f"messages 类型: {AgentState.__annotations__['messages']}")

## 2. 加载手写的 StateGraph

对比 v1（create_agent）和 v2（手写）的结构差异。

In [ ]:
from src.agent.graph import graph

# 查看 graph 的节点和边
print(f"节点: {list(graph.nodes.keys())}")
print(f"类型: {type(graph).__name__}")
print()
# 可视化 graph 结构（需要 graphviz，没有也可看文字描述）
print("Graph 结构:")
print("  [START] → agent → (条件: 有tool_calls?)")
print("                        ├── YES → tools → agent (循环)")
print("                        └── NO  → END")

## 3. 同步调用 — 观察每一步的消息变化

LangGraph 的核心概念是 **thread_id**：
- 同一个 thread_id = 同一段对话
- Checkpointer 用 thread_id 记住对话状态
- 不同的 thread_id 是完全隔离的

In [ ]:
config = {"configurable": {"thread_id": "demo-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "现在几点？"}]},
    config=config,
)

# 遍历最终的消息列表，展示每一步
print("=== 消息流转过程 ===\n")
for i, msg in enumerate(result["messages"]):
    role = getattr(msg, 'type', 'unknown')
    print(f"[步骤 {i}] 角色: {role}")
    print(f"       内容: {msg.content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"       >>> 决定调用工具: {tc['name']}{tc['args']}")
    print()

## 4. 带计算的调用 — 多轮工具调用

同一个对话中可能涉及多轮工具调用（理论上无限循环，直到 LLM 觉得够了）。

In [ ]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "先告诉我现在几点，然后帮我算 365 × 24 × 60 × 60"}]},
    config={"configurable": {"thread_id": "demo-2"}},
)

tool_count = 0
for i, msg in enumerate(result["messages"]):
    role = getattr(msg, 'type', 'unknown')
    if role == "tool":
        tool_count += 1
        print(f"[工具调用 {tool_count}] {msg.content[:100]}..." if len(str(msg.content)) > 100 else f"[工具调用 {tool_count}] {msg.content}")
    elif role == "ai":
        content = str(msg.content) if msg.content else "(无文本，只有工具调用决策)"
        print(f"[AI 回复] {content[:200]}..." if len(content) > 200 else f"[AI 回复] {content}")
    elif role == "human":
        print(f"[用户输入] {msg.content}")
    print()

## 5. 多轮对话 — Checkpointer 的作用

同一个 thread_id，发送第二条消息时，Agent 记得上一轮的对话内容。

In [ ]:
config = {"configurable": {"thread_id": "multi-turn-demo"}}

# 第一轮
graph.invoke(
    {"messages": [{"role": "user", "content": "我叫小明"}]},
    config=config,
)

# 第二轮 —— 同一个 thread_id，Agent 应该"记得"我叫小明
result = graph.invoke(
    {"messages": [{"role": "user", "content": "我叫什么名字？"}]},
    config=config,
)

# 只打印最后一轮的 AI 回复
for msg in reversed(result["messages"]):
    if getattr(msg, 'type', '') == "ai" and msg.content:
        print(f"Agent 回答: {msg.content}")
        break

## 6. 对比 v1 和 v2 的行为

v1 (`create_agent()`) 和 v2 (手写 StateGraph) 的**运行时行为完全一致**，
因为 v1 底层也是建一个 StateGraph。区别在于：

| | v1 create_agent() | v2 手写 StateGraph |
|---|---|---|
| 代码量 | 一行 | ~50 行 |
| 可控性 | 只能通过 middleware 扩展 | 完全掌控每个节点 |
| 适合场景 | 简单 Agent、快速原型 | 复杂多步骤、自定义分支 |
| 底层原理 | 封装了 StateGraph | 直接操作 StateGraph |

## 本节小结

ReAct 循环的核心只有 4 样东西：

| 组件 | 作用 |
|------|------|
| **State** (TypedDict) | 节点间的共享数据，messages 是灵魂 |
| **agent 节点** | 调 LLM，把对话历史变成 AIMessage |
| **tools 节点** (ToolNode) | 自动执行 AIMessage 中的工具调用 |
| **条件边** (tools_condition) | 根据是否有 tool_calls 决定循环还是结束 |

**下一步**：在 `03_middleware.ipynb` 中加入中间件，看看怎么在不改 graph 的前提下注入自定义逻辑。